# NFLX Insight — Netflix-stock-analysis

Pipeline completo: coleta → auditoria → EDA → características → modelos → avaliação → interpretação → dashboard.

Execute de cima para baixo no Colab. Yahoo e SEC são fontes externas sujeitas a bloqueios e revisões. A SEC solicita identificação de contato; o Drive solicita autorização. As métricas históricas documentadas correspondem à execução de 13/10/2026; nova coleta pode produzir valores diferentes.

A/B neste projeto é comparação histórica pareada, sem randomização nem inferência causal. Não houve teste A/B com usuários.


## 1. Ambiente e coleta
Instalação somente no runtime do Colab. As versões efetivamente utilizadas são exportadas ao final.

In [ ]:
%pip -q install pandas_market_calendars


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import io
import json
import platform
import requests
import pandas as pd

ROOT = Path("/content/Netflix-stock-analysis")
RAW = ROOT / "data" / "raw"
REPORTS = ROOT / "reports"

for folder in (RAW, REPORTS):
    folder.mkdir(parents=True, exist_ok=True)

import shutil
from IPython.display import display

manifest = []
raw_frames = {}

def preserve_response(response, filename, source, parameters=None):
    """Preserva os bytes recebidos antes de qualquer transformação."""
    response.raise_for_status()
    payload = response.content

    if not payload:
        raise ValueError(f"Resposta vazia: {source}")

    digest = hashlib.sha256(payload).hexdigest()
    path = RAW / f"{Path(filename).stem}_{digest[:12]}{Path(filename).suffix}"

    if not path.exists():
        path.write_bytes(payload)

    manifest.append({
        "source": source,
        "parameters": parameters or {},
        "collected_at_utc": datetime.now(timezone.utc).isoformat(),
        "file": str(path.relative_to(ROOT)),
        "sha256": digest,
        "bytes": len(payload),
        "http_status": response.status_code,
        "content_type": response.headers.get("Content-Type"),
    })

    (REPORTS / "collection_manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    return path

COMMIT = "ba64c7b32b35bae586dba623dca7746af4596f49"
CSV_URL = (
    "https://raw.githubusercontent.com/"
    "AlexandreSantosRodrigues/Netflix-stock-analysis-powerbi/"
    f"{COMMIT}/NFLX.csv"
)

response = requests.get(CSV_URL, timeout=60)
original_path = preserve_response(
    response,
    "nflx_original.csv",
    CSV_URL,
    {"git_commit": COMMIT},
)

original = pd.read_csv(
    io.BytesIO(original_path.read_bytes()),
    dtype=str,
    keep_default_na=False,
)
raw_frames["original_csv"] = original

environment = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "requests": requests.__version__,
}
(REPORTS / "environment.json").write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

print("Snapshot preservado:", original_path.name)
print("Dimensão:", original.shape)
display(original.head())

In [ ]:
import time

START = "2018-01-01"
END_EXCLUSIVE = "2026-10-13"

def unix_timestamp(date_string):
    return int(
        datetime.strptime(date_string, "%Y-%m-%d")
        .replace(tzinfo=timezone.utc)
        .timestamp()
    )

collection_status = []

for ticker, filename in [
    ("NFLX", "yahoo_nflx.json"),
    ("^GSPC", "yahoo_sp500.json"),
]:
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
    parameters = {
        "period1": unix_timestamp(START),
        "period2": unix_timestamp(END_EXCLUSIVE),
        "interval": "1d",
        "events": "div,splits",
        "includeAdjustedClose": "true",
    }

    try:
        response = requests.get(
            url,
            params=parameters,
            headers={"User-Agent": "NetflixStockAnalysis/1.0"},
            timeout=60,
        )

        path = preserve_response(
            response, filename, url, parameters
        )

        document = json.loads(path.read_bytes())
        chart = document.get("chart", {})

        if chart.get("error"):
            raise ValueError(str(chart["error"]))

        results = chart.get("result")
        if not results or not results[0].get("timestamp"):
            raise ValueError("Resposta sem observações históricas.")

        collection_status.append({
            "ticker": ticker,
            "status": "success",
            "file": str(path.relative_to(ROOT)),
            "observations": len(results[0]["timestamp"]),
            "error": None,
        })

    except (requests.RequestException, ValueError, KeyError) as error:
        collection_status.append({
            "ticker": ticker,
            "status": "failed",
            "file": None,
            "observations": None,
            "error": str(error),
        })

    time.sleep(1)

status = pd.DataFrame(collection_status)
status.to_csv(REPORTS / "collection_status.csv", index=False)
display(status)

In [ ]:
def parse_yahoo_snapshot(relative_path, ticker):
    document = json.loads((ROOT / relative_path).read_bytes())
    result = document["chart"]["result"][0]

    timestamps = result["timestamp"]
    quote = result["indicators"]["quote"][0]
    adjusted = result["indicators"].get("adjclose", [{}])[0]
    exchange_timezone = result["meta"].get("exchangeTimezoneName")

    if not exchange_timezone:
        raise ValueError(f"{ticker}: fuso da bolsa ausente.")

    dates = (
        pd.to_datetime(timestamps, unit="s", utc=True)
        .tz_convert(exchange_timezone)
        .tz_localize(None)
        .normalize()
    )

    def field(values, name):
        if values is None:
            return [None] * len(timestamps)
        if len(values) != len(timestamps):
            raise ValueError(f"{ticker}: tamanho inconsistente em {name}.")
        return values

    frame = pd.DataFrame({
        "Date": dates,
        "Open": field(quote.get("open"), "open"),
        "High": field(quote.get("high"), "high"),
        "Low": field(quote.get("low"), "low"),
        "Close": field(quote.get("close"), "close"),
        "Adj Close": field(adjusted.get("adjclose"), "adjclose"),
        "Volume": field(quote.get("volume"), "volume"),
    })

    for column in frame.columns.drop("Date"):
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    events = []
    for event_type, event_mapping in result.get("events", {}).items():
        for event in event_mapping.values():
            events.append({
                "ticker": ticker,
                "event_type": event_type,
                **event,
            })

    return frame, events

market_frames = {}
corporate_events = []

for item in collection_status:
    if item["status"] != "success":
        continue

    frame, events = parse_yahoo_snapshot(item["file"], item["ticker"])
    market_frames[item["ticker"]] = frame
    corporate_events.extend(events)

assert {"NFLX", "^GSPC"}.issubset(market_frames), \
    "As duas séries precisam estar disponíveis."

events_table = pd.DataFrame(corporate_events)

for ticker, frame in market_frames.items():
    print(
        ticker,
        "| linhas:", len(frame),
        "| início:", frame["Date"].min().date(),
        "| fim:", frame["Date"].max().date(),
    )

display(market_frames["NFLX"].head())
display(events_table)

In [ ]:
def audit_market(frame, source):
    ohlc = ["Open", "High", "Low", "Close"]
    numeric = ohlc + ["Adj Close", "Volume"]
    valid_ohlc = frame[ohlc].notna().all(axis=1)

    invalid_ohlc = valid_ohlc & (
        (frame["High"] < frame[["Open", "Close"]].max(axis=1))
        | (frame["Low"] > frame[["Open", "Close"]].min(axis=1))
        | (frame["High"] < frame["Low"])
    )

    return {
        "source": source,
        "rows": len(frame),
        "duplicate_dates": int(frame["Date"].duplicated().sum()),
        "missing_dates": int(frame["Date"].isna().sum()),
        "ascending_dates": frame["Date"].is_monotonic_increasing,
        "weekend_dates": int((frame["Date"].dt.dayofweek >= 5).sum()),
        "rows_with_missing_numeric": int(
            frame[numeric].isna().any(axis=1).sum()
        ),
        "nonpositive_price_rows": int(
            (frame[ohlc + ["Adj Close"]] <= 0).any(axis=1).sum()
        ),
        "negative_volume": int((frame["Volume"] < 0).sum()),
        "zero_volume": int((frame["Volume"] == 0).sum()),
        "inconsistent_ohlc": int(invalid_ohlc.sum()),
    }

market_audit = pd.DataFrame([
    audit_market(frame, ticker)
    for ticker, frame in market_frames.items()
])
market_audit.to_csv(REPORTS / "market_audit.csv", index=False)
display(market_audit)

nflx_dates = set(market_frames["NFLX"]["Date"].dropna())
sp500_dates = set(market_frames["^GSPC"]["Date"].dropna())

date_alignment = pd.DataFrame({
    "check": ["only_NFLX", "only_SP500", "shared_dates"],
    "count": [
        len(nflx_dates - sp500_dates),
        len(sp500_dates - nflx_dates),
        len(nflx_dates & sp500_dates),
    ],
})
display(date_alignment)

for ticker, frame in market_frames.items():
    incomplete = frame[
        frame[["Open", "High", "Low", "Close", "Adj Close", "Volume"]]
        .isna().any(axis=1)
    ]
    if not incomplete.empty:
        print(f"Observações incompletas — {ticker}")
        display(incomplete.head(10))

In [ ]:
old = original.copy()
old["Date"] = pd.to_datetime(
    old["Date"], format="%Y-%m-%d", errors="coerce"
)

for column in ["Close", "Adj Close", "Volume"]:
    old[column] = pd.to_numeric(old[column], errors="coerce")

new = market_frames["NFLX"]

# O relacionamento por data deve ser um para um.
comparison = old[["Date", "Close", "Adj Close", "Volume"]].merge(
    new[["Date", "Close", "Adj Close", "Volume"]],
    on="Date",
    how="inner",
    suffixes=("_original", "_current"),
    validate="one_to_one",
).sort_values("Date")

if comparison.empty:
    raise ValueError("Não há datas em comum entre as fontes.")

comparison["close_scale_ratio"] = (
    comparison["Close_original"] / comparison["Close_current"]
)
comparison["volume_scale_ratio"] = (
    comparison["Volume_current"] / comparison["Volume_original"]
)

# Retornos são calculados em cada série antes do alinhamento.
old_returns = old.sort_values("Date").set_index("Date")[
    "Adj Close"
].pct_change(fill_method=None)

new_returns = new.sort_values("Date").set_index("Date")[
    "Adj Close"
].pct_change(fill_method=None)

returns_comparison = pd.concat(
    [
        old_returns.rename("return_original"),
        new_returns.rename("return_current"),
    ],
    axis=1,
    join="inner",
)

returns_comparison["absolute_difference_bps"] = (
    (
        returns_comparison["return_original"]
        - returns_comparison["return_current"]
    ).abs() * 10_000
)

print("Datas em comum:", len(comparison))
print("\nRazões de escala — apenas diagnóstico:")
display(
    comparison[["close_scale_ratio", "volume_scale_ratio"]]
    .describe()
)

print("\nDiferença absoluta dos retornos, em pontos-base:")
display(returns_comparison["absolute_difference_bps"].describe())

print("\nMaiores divergências:")
display(
    returns_comparison.nlargest(10, "absolute_difference_bps")
)

comparison.to_csv(REPORTS / "source_reconciliation.csv", index=False)
returns_comparison.to_csv(REPORTS / "return_reconciliation.csv")

In [ ]:
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

nflx = market_frames["NFLX"].copy()
sp500 = market_frames["^GSPC"].copy()

def validate_prices(frame, source):
    required_prices = ["Open", "High", "Low", "Close", "Adj Close"]

    assert frame["Date"].notna().all(), f"{source}: datas ausentes."
    assert not frame["Date"].duplicated().any(), \
        f"{source}: datas duplicadas."
    assert frame[required_prices].notna().all().all(), \
        f"{source}: preços ausentes."
    assert (frame[required_prices] > 0).all().all(), \
        f"{source}: preços não positivos."

    consistent = (
        (frame["High"] >= frame[["Open", "Close"]].max(axis=1))
        & (frame["Low"] <= frame[["Open", "Close"]].min(axis=1))
        & (frame["High"] >= frame["Low"])
    )
    assert consistent.all(), f"{source}: OHLC inconsistente."

validate_prices(nflx, "NFLX")
validate_prices(sp500, "S&P 500")

assert nflx["Volume"].notna().all(), "NFLX: volume ausente."
assert (nflx["Volume"] > 0).all(), "NFLX: volume não positivo."

print("Observações de volume zero no índice:")
display(sp500.loc[sp500["Volume"].eq(0)])

base = nflx.merge(
    sp500[["Date", "Adj Close"]].rename(
        columns={"Adj Close": "SP500_Adj_Close"}
    ),
    on="Date",
    how="outer",
    validate="one_to_one",
    indicator=True,
).sort_values("Date").reset_index(drop=True)

assert base["_merge"].eq("both").all(), \
    "Existem datas sem correspondência entre as séries."

base = base.drop(columns="_merge")

base["NFLX_Return"] = base["Adj Close"].pct_change(fill_method=None)
base["SP500_Return"] = base["SP500_Adj_Close"].pct_change(fill_method=None)

# Diferença diária de retornos; não representa alfa ajustado ao risco.
base["Excess_Return"] = base["NFLX_Return"] - base["SP500_Return"]

base["NFLX_Drawdown"] = (
    base["Adj Close"] / base["Adj Close"].cummax() - 1
)
base["SP500_Drawdown"] = (
    base["SP500_Adj_Close"] / base["SP500_Adj_Close"].cummax() - 1
)

base.to_csv(PROCESSED / "market_base.csv", index=False)

decisions = {
    "primary_source": "Yahoo Finance snapshots",
    "legacy_csv": "Preserved separately for reconciliation",
    "price_basis": "Provider-adjusted historical series",
    "additional_split_adjustment": False,
    "benchmark": "S&P 500 price index; excludes reinvested dividends",
    "benchmark_volume": "Excluded from features",
    "missing_price_imputation": False,
    "outlier_removal": False,
    "return_definition": "Adjusted close percentage change",
}
(REPORTS / "cleaning_decisions.json").write_text(
    json.dumps(decisions, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Base validada:", base.shape)
display(base.head())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

TRADING_DAYS = 252

def performance_summary(prices, returns, drawdown):
    valid_returns = returns.dropna()
    elapsed_years = (
        base["Date"].iloc[-1] - base["Date"].iloc[0]
    ).days / 365.25

    total_return = prices.iloc[-1] / prices.iloc[0] - 1

    return {
        "Retorno acumulado (%)": total_return * 100,
        "Crescimento anualizado (%)": (
            (1 + total_return) ** (1 / elapsed_years) - 1
        ) * 100,
        "Volatilidade anualizada (%)": (
            valid_returns.std(ddof=1) * np.sqrt(TRADING_DAYS) * 100
        ),
        "Drawdown máximo (%)": drawdown.min() * 100,
        "Pior sessão (%)": valid_returns.min() * 100,
        "Melhor sessão (%)": valid_returns.max() * 100,
    }

summary = pd.DataFrame({
    "NFLX": performance_summary(
        base["Adj Close"], base["NFLX_Return"], base["NFLX_Drawdown"]
    ),
    "S&P 500": performance_summary(
        base["SP500_Adj_Close"],
        base["SP500_Return"],
        base["SP500_Drawdown"],
    ),
}).T

summary.to_csv(REPORTS / "performance_summary.csv")
display(summary.round(2))

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for column, label in [
    ("Adj Close", "NFLX"),
    ("SP500_Adj_Close", "S&P 500"),
]:
    normalized = base[column] / base[column].iloc[0] * 100
    axes[0].plot(base["Date"], normalized, label=label)

axes[0].set_title("Evolução dos preços — base 100")
axes[0].set_ylabel("Índice")
axes[0].legend()

axes[1].plot(
    base["Date"], base["NFLX_Drawdown"] * 100, label="NFLX"
)
axes[1].plot(
    base["Date"], base["SP500_Drawdown"] * 100, label="S&P 500"
)
axes[1].set_title("Queda em relação ao maior preço observado")
axes[1].set_ylabel("Drawdown (%)")
axes[1].legend()

for column, label in [
    ("NFLX_Return", "NFLX"),
    ("SP500_Return", "S&P 500"),
]:
    rolling_volatility = (
        base[column].rolling(30, min_periods=30).std()
        * np.sqrt(TRADING_DAYS) * 100
    )
    axes[2].plot(base["Date"], rolling_volatility, label=label)

axes[2].set_title("Volatilidade anualizada — janela de 30 pregões")
axes[2].set_ylabel("Volatilidade (%)")
axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(REPORTS / "market_overview.png", dpi=150)
plt.show()

In [ ]:
extreme_sessions = (
    base.assign(Absolute_Return=base["NFLX_Return"].abs())
    .nlargest(10, "Absolute_Return")
    [[
        "Date", "Close", "Volume",
        "NFLX_Return", "SP500_Return", "Excess_Return"
    ]]
    .copy()
)

print("Maiores movimentos diários da NFLX:")
display(extreme_sessions)

volume_review = comparison[
    ["Date", "Volume_original", "Volume_current", "volume_scale_ratio"]
].copy()

volume_review["difference_after_scale_pct"] = (
    volume_review["Volume_current"]
    / (volume_review["Volume_original"] * 10) - 1
) * 100

print("Maiores diferenças de volume após considerar a escala 10:1:")
display(
    volume_review.assign(
        absolute_difference=lambda x:
            x["difference_after_scale_pct"].abs()
    )
    .nlargest(10, "absolute_difference")
    .drop(columns="absolute_difference")
)

extreme_sessions.to_csv(REPORTS / "extreme_sessions.csv", index=False)
volume_review.to_csv(REPORTS / "volume_reconciliation.csv", index=False)

## 2. Características e validação temporal
Previsão após o fechamento para o retorno até o próximo fechamento. Características do dia corrente estão disponíveis nesse instante. Snapshot retrospectivamente ajustado não é um histórico point-in-time.

In [ ]:
features = base.copy().sort_values("Date").reset_index(drop=True)

features["return_today"] = features["NFLX_Return"]
features["market_return_today"] = features["SP500_Return"]
features["excess_return_today"] = features["Excess_Return"]

for lag in [1, 2, 5]:
    features[f"return_lag_{lag}"] = features["NFLX_Return"].shift(lag)

for window in [5, 20, 60]:
    features[f"momentum_{window}"] = (
        features["Adj Close"]
        / features["Adj Close"].shift(window) - 1
    )

    features[f"volatility_{window}"] = (
        features["NFLX_Return"]
        .rolling(window, min_periods=window)
        .std()
    )

    moving_average = features["Adj Close"].rolling(
        window, min_periods=window
    ).mean()

    features[f"distance_ma_{window}"] = (
        features["Adj Close"] / moving_average - 1
    )

features["relative_volume_20"] = (
    features["Volume"]
    / features["Volume"].rolling(20, min_periods=20).mean()
)

features["intraday_range"] = (
    (features["High"] - features["Low"]) / features["Close"]
)
features["intraday_return"] = (
    features["Close"] / features["Open"] - 1
)

features["market_momentum_20"] = (
    features["SP500_Adj_Close"]
    / features["SP500_Adj_Close"].shift(20) - 1
)

rolling_covariance = (
    features["NFLX_Return"]
    .rolling(60, min_periods=60)
    .cov(features["SP500_Return"])
)
rolling_market_variance = (
    features["SP500_Return"]
    .rolling(60, min_periods=60)
    .var()
)

features["beta_60"] = (
    rolling_covariance
    / rolling_market_variance.replace(0, np.nan)
)

features["target_return"] = features["NFLX_Return"].shift(-1)
features["target_date"] = features["Date"].shift(-1)

feature_columns = [
    "return_today",
    "market_return_today",
    "excess_return_today",
    "return_lag_1",
    "return_lag_2",
    "return_lag_5",
    *[
        f"{prefix}_{window}"
        for window in [5, 20, 60]
        for prefix in ["momentum", "volatility", "distance_ma"]
    ],
    "relative_volume_20",
    "intraday_range",
    "intraday_return",
    "market_momentum_20",
    "beta_60",
]

features[feature_columns] = features[feature_columns].replace(
    [np.inf, -np.inf], np.nan
)

complete_features = features[feature_columns].notna().all(axis=1)
known_target = (
    features["target_return"].notna()
    & features["target_date"].notna()
)

dataset = features.loc[
    complete_features & known_target,
    ["Date", "target_date", *feature_columns, "target_return"]
].copy()

inference = features.loc[
    complete_features & ~known_target,
    ["Date", *feature_columns]
].copy()

dataset.to_csv(PROCESSED / "model_dataset.csv", index=False)
inference.to_csv(PROCESSED / "inference_features.csv", index=False)

print("Características:", len(feature_columns))
print("Linhas para modelagem:", len(dataset))
print("Linhas sem características completas:", int((~complete_features).sum()))
print("Linhas para inferência:", len(inference))

In [ ]:
train = dataset.loc[
    dataset["target_date"] < pd.Timestamp("2024-01-01")
].copy()

validation = dataset.loc[
    (dataset["target_date"] >= pd.Timestamp("2024-01-01"))
    & (dataset["target_date"] < pd.Timestamp("2025-01-01"))
].copy()

test = dataset.loc[
    dataset["target_date"] >= pd.Timestamp("2025-01-01")
].copy()

assert all(len(part) > 0 for part in [train, validation, test])
assert train["target_date"].max() < validation["target_date"].min()
assert validation["target_date"].max() < test["target_date"].min()
assert (dataset["Date"] < dataset["target_date"]).all()

split_summary = pd.DataFrame([
    {
        "set": name,
        "rows": len(part),
        "first_target_date": part["target_date"].min(),
        "last_target_date": part["target_date"].max(),
    }
    for name, part in [
        ("train", train),
        ("validation", validation),
        ("test", test),
    ]
])

display(split_summary)
split_summary.to_csv(REPORTS / "temporal_split.csv", index=False)

protocol = {
    "prediction_time": "After session close",
    "target": "Next-session adjusted close-to-close return",
    "split_basis": "Target observation date",
    "baseline": "Zero return, equivalent to unchanged next closing price",
    "candidate_models": ["Ridge", "HistGradientBoostingRegressor"],
    "primary_metric": "MAE of returns",
    "secondary_metrics": ["RMSE of returns", "Directional accuracy"],
    "selection_data": "Training temporal folds and 2024 validation",
    "final_test": "2025 onward; no tuning on this period",
    "limitation": "Historical prices adjusted using current snapshot",
}

(REPORTS / "evaluation_protocol.json").write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Protocolo registrado. Nenhum modelo foi avaliado no teste.")

In [ ]:
from pandas.testing import assert_frame_equal

assert not dataset["Date"].duplicated().any()
assert np.isfinite(dataset[feature_columns].to_numpy()).all()
assert np.isfinite(dataset["target_return"].to_numpy()).all()

expected_target = base["NFLX_Return"].shift(-1)
np.testing.assert_allclose(
    features["target_return"].to_numpy(),
    expected_target.to_numpy(),
    equal_nan=True,
)


cutoff_position = len(base) // 2
prefix = base.iloc[:cutoff_position + 1].copy()

expected_momentum = (
    prefix["Adj Close"] / prefix["Adj Close"].shift(20) - 1
)
expected_volatility = (
    prefix["NFLX_Return"].rolling(20, min_periods=20).std()
)

np.testing.assert_allclose(
    features.loc[:cutoff_position, "momentum_20"],
    expected_momentum,
    equal_nan=True,
)
np.testing.assert_allclose(
    features.loc[:cutoff_position, "volatility_20"],
    expected_volatility,
    equal_nan=True,
)

assert_frame_equal(
    features[base.columns].reset_index(drop=True),
    base.reset_index(drop=True),
)

print("Testes de integridade passaram.")

In [ ]:
import sklearn
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42

X_train = train[feature_columns]
y_train = train["target_return"]

X_validation = validation[feature_columns]
y_validation = validation["target_return"]

temporal_cv = TimeSeriesSplit(n_splits=5, gap=1)

fold_summary = []

for fold, (fit_idx, eval_idx) in enumerate(
    temporal_cv.split(X_train), start=1
):
    fit_part = train.iloc[fit_idx]
    eval_part = train.iloc[eval_idx]

    assert (
        fit_part["target_date"].max()
        < eval_part["Date"].min()
    ), "Um alvo de treino invade a janela de previsão."

    fold_summary.append({
        "fold": fold,
        "train_rows": len(fit_idx),
        "evaluation_rows": len(eval_idx),
        "last_training_target": fit_part["target_date"].max(),
        "first_prediction_date": eval_part["Date"].min(),
        "last_evaluation_target": eval_part["target_date"].max(),
    })

display(pd.DataFrame(fold_summary))

environment["scikit-learn"] = sklearn.__version__
(REPORTS / "environment.json").write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

In [ ]:
ridge_search = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge()),
    ]),
    param_grid={
        "model__alpha": [0.1, 1.0, 10.0, 100.0, 1000.0]
    },
    scoring="neg_mean_absolute_error",
    cv=temporal_cv,
    n_jobs=2,
    refit=True,
    error_score="raise",
)

boosting_search = GridSearchCV(
    estimator=HistGradientBoostingRegressor(
        loss="squared_error",
        max_iter=150,
        early_stopping=False,
        random_state=SEED,
    ),
    param_grid={
        "learning_rate": [0.03, 0.1],
        "max_leaf_nodes": [7, 15],
        "min_samples_leaf": [30, 60],
        "l2_regularization": [1.0, 10.0],
    },
    scoring="neg_mean_absolute_error",
    cv=temporal_cv,
    n_jobs=2,
    refit=True,
    error_score="raise",
)

searches = {
    "Ridge": ridge_search,
    "HistGradientBoosting": boosting_search,
}

cv_results = []

# Baseline nos mesmos blocos temporais dos modelos.
baseline_fold_mae = []
for _, eval_idx in temporal_cv.split(X_train):
    baseline_fold_mae.append(
        mean_absolute_error(
            y_train.iloc[eval_idx],
            np.zeros(len(eval_idx)),
        )
    )

cv_results.append({
    "model": "Baseline_zero",
    "cv_mae_bps": np.mean(baseline_fold_mae) * 10_000,
    "cv_fold_std_bps": np.std(baseline_fold_mae) * 10_000,
    "parameters": "{}",
})

for name, search in searches.items():
    print(f"Ajustando {name}...")
    search.fit(X_train, y_train)

    cv_results.append({
        "model": name,
        "cv_mae_bps": -search.best_score_ * 10_000,
        "cv_fold_std_bps": (
            search.cv_results_["std_test_score"][search.best_index_]
            * 10_000
        ),
        "parameters": json.dumps(search.best_params_),
    })

    pd.DataFrame(search.cv_results_).to_csv(
        REPORTS / f"{name.lower()}_cv_results.csv",
        index=False,
    )

cv_summary = pd.DataFrame(cv_results).sort_values("cv_mae_bps")
cv_summary.to_csv(REPORTS / "cv_summary.csv", index=False)
display(cv_summary)

In [ ]:
validation_predictions = {
    "Baseline_zero": np.zeros(len(validation)),
    **{
        name: search.best_estimator_.predict(X_validation)
        for name, search in searches.items()
    },
}

def regression_metrics(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    directional_accuracy = (
        np.mean(np.sign(actual) == np.sign(predicted)) * 100
        if np.any(predicted != 0)
        else np.nan
    )

    return {
        "MAE_bps": mean_absolute_error(actual, predicted) * 10_000,
        "RMSE_bps": np.sqrt(
            mean_squared_error(actual, predicted)
        ) * 10_000,
        "bias_bps": np.mean(predicted - actual) * 10_000,
        "directional_accuracy_pct": directional_accuracy,
    }

validation_results = pd.DataFrame([
    {
        "model": name,
        **regression_metrics(y_validation, prediction),
    }
    for name, prediction in validation_predictions.items()
])

baseline_mae = validation_results.loc[
    validation_results["model"].eq("Baseline_zero"), "MAE_bps"
].iloc[0]

validation_results["MAE_improvement_vs_baseline_pct"] = (
    1 - validation_results["MAE_bps"] / baseline_mae
) * 100

validation_results = validation_results.sort_values("MAE_bps")
validation_results.to_csv(
    REPORTS / "validation_metrics.csv", index=False
)

validation_output = validation[
    ["Date", "target_date", "target_return"]
].copy()

for name, prediction in validation_predictions.items():
    validation_output[f"prediction_{name}"] = prediction

validation_output.to_csv(
    REPORTS / "validation_predictions.csv", index=False
)

display(validation_results.round(3))
print("Valores positivos de melhoria indicam redução do MAE.")
print("O conjunto de teste continua sem avaliação.")

In [ ]:
error_rows = []

for name, prediction in validation_predictions.items():
    diagnostics = validation[
        ["target_date", "target_return"]
    ].copy()

    diagnostics["prediction"] = prediction
    diagnostics["absolute_error_bps"] = (
        diagnostics["prediction"] - diagnostics["target_return"]
    ).abs() * 10_000

    diagnostics["quarter"] = (
        diagnostics["target_date"].dt.to_period("Q").astype(str)
    )

    quarterly = diagnostics.groupby("quarter").agg(
        observations=("absolute_error_bps", "size"),
        MAE_bps=("absolute_error_bps", "mean"),
        largest_error_bps=("absolute_error_bps", "max"),
    ).reset_index()

    quarterly.insert(0, "model", name)
    error_rows.append(quarterly)

    print(f"\nCinco maiores erros — {name}")
    display(
        diagnostics.nlargest(5, "absolute_error_bps")[
            ["target_date", "target_return",
             "prediction", "absolute_error_bps"]
        ]
    )

quarterly_errors = pd.concat(error_rows, ignore_index=True)
quarterly_errors.to_csv(
    REPORTS / "validation_quarterly_errors.csv", index=False
)
display(quarterly_errors.round(3))

## 3. Protocolo congelado, teste e comparação histórica
Baseline selecionado em 2024. Ridge experimental reajustada antes do teste; nenhum ajuste durante o teste. Custos de 5 bps por ponta. As duas estratégias perderam dinheiro no cenário principal. Intervalo da diferença média inclui zero.

In [ ]:
frozen_protocol = {
    **protocol,
    "selected_model": "Baseline_zero",
    "experimental_model": "Ridge",
    "selection_reason": "Lowest MAE on 2024 validation",
    "test_model_update": "One refit before test; no updates during test",
    "strategy_A": "Buy next open and sell same-session close every day",
    "strategy_B": "Same trade only if predicted return > 0.001",
    "signal_threshold": 0.001,
    "cost_per_side": 0.0005,
    "cash_return": 0.0,
    "leverage": False,
    "experiment_type": "Historical paired comparison; not causal A/B",
    "strategy_target_limitation": (
        "Model predicts close-to-close; execution earns open-to-close"
    ),
    "bootstrap_block_length": 10,
    "bootstrap_repetitions": 2000,
}

protocol_path = REPORTS / "frozen_evaluation_protocol.json"
protocol_path.write_text(
    json.dumps(frozen_protocol, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Protocolo congelado.")
print("Modelo principal: baseline. Alternativa experimental: Ridge.")

In [ ]:
from sklearn.base import clone

first_test_prediction_date = test["Date"].min()

development = dataset.loc[
    dataset["target_date"] < first_test_prediction_date
].copy()

assert development["target_date"].max() < first_test_prediction_date
assert set(development.index).isdisjoint(set(test.index))

final_ridge = clone(ridge_search.best_estimator_)
final_ridge.fit(
    development[feature_columns],
    development["target_return"],
)

test_predictions = {
    "Baseline_zero": np.zeros(len(test)),
    "Ridge": final_ridge.predict(test[feature_columns]),
}

test_metrics = pd.DataFrame([
    {
        "model": name,
        **regression_metrics(test["target_return"], prediction),
    }
    for name, prediction in test_predictions.items()
])

test_baseline_mae = test_metrics.loc[
    test_metrics["model"].eq("Baseline_zero"), "MAE_bps"
].iloc[0]

test_metrics["MAE_improvement_vs_baseline_pct"] = (
    1 - test_metrics["MAE_bps"] / test_baseline_mae
) * 100

test_output = test[
    ["Date", "target_date", "target_return"]
].copy()

for name, prediction in test_predictions.items():
    test_output[f"prediction_{name}"] = prediction

test_metrics.to_csv(REPORTS / "test_metrics.csv", index=False)
test_output.to_csv(REPORTS / "test_predictions.csv", index=False)

display(test_metrics.round(3))
print("A seleção permanece baseline, independentemente desta tabela.")

In [ ]:
execution = test_output.merge(
    base[["Date", "Open", "Close"]].rename(
        columns={"Date": "target_date"}
    ),
    on="target_date",
    how="left",
    validate="one_to_one",
).sort_values("target_date").reset_index(drop=True)

assert execution[["Open", "Close"]].notna().all().all()
assert (execution["Date"] < execution["target_date"]).all()

COST = frozen_protocol["cost_per_side"]
THRESHOLD = frozen_protocol["signal_threshold"]

execution["intraday_asset_return"] = (
    execution["Close"] / execution["Open"] - 1
)

# Custo proporcional em cada ponta da operação.
execution["net_trade_return"] = (
    (execution["Close"] / execution["Open"])
    * (1 - COST) / (1 + COST) - 1
)

execution["position_A"] = 1
execution["position_B"] = (
    execution["prediction_Ridge"] > THRESHOLD
).astype(int)

for strategy in ["A", "B"]:
    execution[f"return_{strategy}"] = (
        execution[f"position_{strategy}"]
        * execution["net_trade_return"]
    )
    execution[f"wealth_{strategy}"] = (
        1 + execution[f"return_{strategy}"]
    ).cumprod()

def strategy_summary(strategy):
    returns = execution[f"return_{strategy}"]
    wealth = execution[f"wealth_{strategy}"]

    # Inclui o capital inicial como referência para o drawdown.
    running_peak = wealth.cummax().clip(lower=1.0)
    drawdown = wealth / running_peak - 1

    return {
        "strategy": strategy,
        "sessions": len(execution),
        "trades": int(execution[f"position_{strategy}"].sum()),
        "exposure_pct": execution[f"position_{strategy}"].mean() * 100,
        "net_total_return_pct": (wealth.iloc[-1] - 1) * 100,
        "annualized_volatility_pct": (
            returns.std(ddof=1) * np.sqrt(252) * 100
        ),
        "max_drawdown_pct": drawdown.min() * 100,
        "mean_daily_return_bps": returns.mean() * 10_000,
    }

ab_summary = pd.DataFrame([
    strategy_summary("A"),
    strategy_summary("B"),
])

execution.to_csv(REPORTS / "historical_ab_daily.csv", index=False)
ab_summary.to_csv(REPORTS / "historical_ab_summary.csv", index=False)

display(ab_summary.round(3))

plt.figure(figsize=(12, 5))
for strategy in ["A", "B"]:
    plt.plot(
        execution["target_date"],
        execution[f"wealth_{strategy}"],
        label=f"Estratégia {strategy}",
    )

plt.title("Comparação histórica A/B — capital inicial 1, após custos")
plt.ylabel("Capital acumulado")
plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()
plt.savefig(REPORTS / "historical_ab.png", dpi=150)
plt.show()

In [ ]:
daily_difference = (
    execution["return_B"] - execution["return_A"]
).to_numpy()

rng = np.random.default_rng(SEED)
n = len(daily_difference)
block_length = frozen_protocol["bootstrap_block_length"]
repetitions = frozen_protocol["bootstrap_repetitions"]
blocks_needed = int(np.ceil(n / block_length))

bootstrap_means = np.empty(repetitions)

for iteration in range(repetitions):
    starts = rng.integers(
        0, n - block_length + 1, size=blocks_needed
    )
    indices = np.concatenate([
        np.arange(start, start + block_length)
        for start in starts
    ])[:n]

    bootstrap_means[iteration] = daily_difference[indices].mean()

lower, upper = np.quantile(bootstrap_means, [0.025, 0.975])

ab_uncertainty = pd.DataFrame([{
    "comparison": "B minus A",
    "mean_daily_difference_bps": daily_difference.mean() * 10_000,
    "approximate_95pct_lower_bps": lower * 10_000,
    "approximate_95pct_upper_bps": upper * 10_000,
    "interval_contains_zero": bool(lower <= 0 <= upper),
}])

ab_uncertainty.to_csv(
    REPORTS / "historical_ab_uncertainty.csv", index=False
)
display(ab_uncertainty.round(3))

In [ ]:
# Confere que as posições correspondem à regra congelada.
np.testing.assert_array_equal(
    execution["position_B"].to_numpy(),
    (
        execution["prediction_Ridge"]
        > frozen_protocol["signal_threshold"]
    ).astype(int).to_numpy(),
)

assert execution["position_B"].isin([0, 1]).all()
assert (execution["Date"] < execution["target_date"]).all()

# Em caixa, B deve ter retorno zero.
assert execution.loc[
    execution["position_B"].eq(0), "return_B"
].eq(0).all()

# Custos positivos não podem melhorar o retorno da operação.
assert (
    execution["net_trade_return"]
    <= execution["intraday_asset_return"] + 1e-12
).all()

for strategy in ["A", "B"]:
    expected_wealth = (
        1 + execution[f"return_{strategy}"]
    ).cumprod()

    np.testing.assert_allclose(
        execution[f"wealth_{strategy}"],
        expected_wealth,
    )

decomposition = []

for strategy in ["A", "B"]:
    position = execution[f"position_{strategy}"]

    gross_daily = position * execution["intraday_asset_return"]
    net_daily = execution[f"return_{strategy}"]

    gross_total = (1 + gross_daily).prod() - 1
    net_total = (1 + net_daily).prod() - 1

    decomposition.append({
        "strategy": strategy,
        "trades": int(position.sum()),
        "gross_total_return_pct": gross_total * 100,
        "net_total_return_pct": net_total * 100,
        "cost_drag_percentage_points": (
            gross_total - net_total
        ) * 100,
    })

decomposition.append({
    "strategy": "Cash_zero_return",
    "trades": 0,
    "gross_total_return_pct": 0.0,
    "net_total_return_pct": 0.0,
    "cost_drag_percentage_points": 0.0,
})

cost_decomposition = pd.DataFrame(decomposition)
cost_decomposition.to_csv(
    REPORTS / "strategy_cost_decomposition.csv", index=False
)

display(cost_decomposition.round(3))
print("Testes de execução e contabilização passaram.")

In [ ]:
sensitivity_rows = []

# Cenários ilustrativos, não estimativas de custos reais.
for cost_bps_per_side in [0, 1, 5, 10, 20]:
    cost = cost_bps_per_side / 10_000

    trade_return = (
        (execution["Close"] / execution["Open"])
        * (1 - cost) / (1 + cost) - 1
    )

    for strategy in ["A", "B"]:
        daily_return = (
            execution[f"position_{strategy}"] * trade_return
        )

        sensitivity_rows.append({
            "cost_bps_per_side": cost_bps_per_side,
            "strategy": strategy,
            "total_return_pct": (
                (1 + daily_return).prod() - 1
            ) * 100,
            "mean_daily_return_bps": daily_return.mean() * 10_000,
        })

cost_sensitivity = pd.DataFrame(sensitivity_rows)
cost_sensitivity.to_csv(
    REPORTS / "strategy_cost_sensitivity.csv", index=False
)

display(
    cost_sensitivity.pivot(
        index="cost_bps_per_side",
        columns="strategy",
        values="total_return_pct",
    ).round(3)
)

In [ ]:
ridge_model = final_ridge.named_steps["model"]

coefficient_table = pd.DataFrame({
    "feature": feature_columns,
    "effect_bps_per_standard_deviation": (
        ridge_model.coef_ * 10_000
    ),
})

coefficient_table["absolute_effect"] = (
    coefficient_table["effect_bps_per_standard_deviation"].abs()
)
coefficient_table = coefficient_table.sort_values(
    "absolute_effect", ascending=False
)

coefficient_table.to_csv(
    REPORTS / "ridge_coefficients.csv", index=False
)

display(
    coefficient_table.drop(columns="absolute_effect").round(3)
)

print(
    "Intercepto da Ridge, em bps:",
    round(float(ridge_model.intercept_) * 10_000, 3),
)

top_coefficients = coefficient_table.head(10).sort_values(
    "effect_bps_per_standard_deviation"
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_coefficients["feature"],
    top_coefficients["effect_bps_per_standard_deviation"],
)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Efeito previsto em bps por desvio-padrão")
plt.title("Ridge — dez maiores coeficientes em magnitude")
plt.tight_layout()
plt.savefig(REPORTS / "ridge_coefficients.png", dpi=150)
plt.show()

## 4. Demonstrações contábeis
Mesmo documento para os três indicadores de cada contexto. Q4 derivados de ano menos nove meses, preservando os dois documentos e suas datas.

In [ ]:
from getpass import getpass

contact_email = getpass("E-mail de contato para identificar o acesso à SEC: ").strip()

if "@" not in contact_email or "\n" in contact_email:
    raise ValueError("Informe um e-mail de contato válido.")

SEC_URL = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    "CIK0001065280.json"
)

sec_response = requests.get(
    SEC_URL,
    headers={
        "User-Agent": f"NetflixStockAnalysis {contact_email}",
        "Accept": "application/json",
    },
    timeout=90,
)

del contact_email

sec_path = preserve_response(
    sec_response,
    "sec_netflix_companyfacts.json",
    SEC_URL,
)

sec_document = json.loads(sec_path.read_bytes())

assert int(sec_document["cik"]) == 1065280
assert "facts" in sec_document

print("Empresa:", sec_document.get("entityName"))
print("Snapshot:", sec_path.name)
print("Taxonomias:", list(sec_document["facts"]))

In [ ]:
fact_rows = []

for taxonomy, concepts in sec_document["facts"].items():
    for concept, details in concepts.items():
        for unit, observations in details.get("units", {}).items():
            for observation in observations:
                fact_rows.append({
                    **observation,
                    "taxonomy": taxonomy,
                    "concept": concept,
                    "label": details.get("label"),
                    "unit": unit,
                })

sec_facts = pd.DataFrame(fact_rows)

for column in ["start", "end", "filed"]:
    if column not in sec_facts:
        sec_facts[column] = pd.NaT
    sec_facts[column] = pd.to_datetime(
        sec_facts[column], errors="coerce"
    )

sec_facts["val"] = pd.to_numeric(sec_facts["val"], errors="coerce")
sec_facts["duration_days"] = (
    sec_facts["end"] - sec_facts["start"]
).dt.days + 1

sec_facts.to_csv(
    PROCESSED / "sec_facts_normalized.csv", index=False
)

print("Observações normalizadas:", len(sec_facts))
display(
    sec_facts[
        ["taxonomy", "concept", "unit", "start", "end",
         "filed", "form", "val"]
    ].head(10)
)

In [ ]:
sec_audit = pd.Series({
    "observations": len(sec_facts),
    "concepts": sec_facts[["taxonomy", "concept"]]
        .drop_duplicates().shape[0],
    "missing_values": int(sec_facts["val"].isna().sum()),
    "missing_period_end": int(sec_facts["end"].isna().sum()),
    "missing_filing_date": int(sec_facts["filed"].isna().sum()),
    "negative_duration_rows": int(
        (sec_facts["duration_days"] <= 0).sum()
    ),
    "exact_duplicate_rows": int(sec_facts.duplicated().sum()),
}).to_frame("result")

display(sec_audit)
sec_audit.to_csv(REPORTS / "sec_facts_audit.csv")

candidate_pattern = (
    r"Revenue|SalesRevenue|OperatingIncomeLoss|NetIncomeLoss|"
    r"NetCashProvidedByUsedInOperatingActivities"
)

concept_catalog = (
    sec_facts.loc[
        sec_facts["taxonomy"].eq("us-gaap")
        & sec_facts["concept"].str.contains(
            candidate_pattern, case=False, regex=True, na=False
        )
    ]
    .groupby(["concept", "label", "unit"], dropna=False)
    .agg(
        observations=("val", "size"),
        first_period=("end", "min"),
        last_period=("end", "max"),
    )
    .reset_index()
)

display(concept_catalog)
concept_catalog.to_csv(
    REPORTS / "sec_candidate_concepts.csv", index=False
)

sec_in_scope = sec_facts.loc[
    sec_facts["filed"] < pd.Timestamp(END_EXCLUSIVE)
].copy()

print("Durações mais frequentes em observações monetárias:")
display(
    sec_in_scope.loc[
        sec_in_scope["unit"].eq("USD"), "duration_days"
    ].value_counts().head(12).to_frame("observations")
)

In [ ]:
CONCEPTS = {
    "Revenues": "revenue",
    "OperatingIncomeLoss": "operating_income",
    "NetIncomeLoss": "net_income",
}

financial_facts = sec_in_scope.loc[
    sec_in_scope["taxonomy"].eq("us-gaap")
    & sec_in_scope["unit"].eq("USD")
    & sec_in_scope["concept"].isin(CONCEPTS)
    & sec_in_scope["form"].isin(["10-K", "10-Q", "10-K/A", "10-Q/A"])
    & sec_in_scope["start"].notna()
    & (sec_in_scope["end"] >= pd.Timestamp("2018-01-01"))
].copy()

keys = ["accn", "filed", "start", "end", "concept"]

# Repetições com mesmo valor são aceitáveis; valores distintos
# no mesmo contexto precisam de investigação.
conflicts = (
    financial_facts.groupby(keys)["val"]
    .nunique()
    .loc[lambda values: values > 1]
)

if not conflicts.empty:
    display(conflicts.to_frame("distinct_values"))
    raise ValueError("Existem conflitos no mesmo documento e período.")

financial_facts = financial_facts.drop_duplicates(keys)

financial_periods = (
    financial_facts.pivot(
        index=["accn", "filed", "start", "end"],
        columns="concept",
        values="val",
    )
    .rename(columns=CONCEPTS)
    .reset_index()
)
financial_periods.columns.name = None

financial_periods["duration_days"] = (
    financial_periods["end"] - financial_periods["start"]
).dt.days + 1

metrics = list(CONCEPTS.values())
incomplete = financial_periods[metrics].isna().any(axis=1)

print("Contextos incompletos, excluídos da tabela de indicadores:",
      int(incomplete.sum()))

financial_periods.loc[incomplete].to_csv(
    REPORTS / "sec_incomplete_financial_contexts.csv", index=False
)

complete_periods = financial_periods.loc[~incomplete].copy()

display(
    complete_periods[
        ["accn", "filed", "start", "end", "duration_days", *metrics]
    ].head()
)

In [ ]:
# Trimestres diretamente divulgados, com limites de calendário.
direct_quarters = complete_periods.loc[
    complete_periods["duration_days"].between(80, 100)
    & complete_periods["start"].dt.is_quarter_start
    & complete_periods["end"].dt.is_quarter_end
].copy()

direct_quarters["method"] = "direct_quarter"

# Ano e nove meses: ambos devem vir do mesmo documento.
annual = complete_periods.loc[
    complete_periods["duration_days"].between(350, 380)
    & complete_periods["start"].dt.month.eq(1)
    & complete_periods["start"].dt.day.eq(1)
    & complete_periods["end"].dt.month.eq(12)
    & complete_periods["end"].dt.day.eq(31)
].copy()

nine_months = complete_periods.loc[
    complete_periods["duration_days"].between(260, 285)
    & complete_periods["start"].dt.month.eq(1)
    & complete_periods["start"].dt.day.eq(1)
    & complete_periods["end"].dt.month.eq(9)
    & complete_periods["end"].dt.day.eq(30)
].copy()



In [ ]:
derived_q4_rows = []
q4_issues = []

annual_selected = (
    annual.sort_values(["end", "filed", "accn"])
    .drop_duplicates(["start", "end"], keep="first")
)

for _, annual_row in annual_selected.iterrows():
    year = annual_row["end"].year

    candidates = nine_months.loc[
        nine_months["start"].eq(annual_row["start"])
        & nine_months["end"].eq(pd.Timestamp(year, 9, 30))
        & (nine_months["filed"] <= annual_row["filed"])
    ].copy()

    if candidates.empty:
        q4_issues.append({
            "year": year,
            "issue": "Nine-month disclosure unavailable by annual filing",
        })
        continue

    nine_row = candidates.sort_values(
        ["filed", "accn"]
    ).iloc[-1]

    derived_q4_rows.append({
        "accn": annual_row["accn"],
        "filed": annual_row["filed"],
        "start": pd.Timestamp(year, 10, 1),
        "end": annual_row["end"],
        **{
            metric: annual_row[metric] - nine_row[metric]
            for metric in metrics
        },
        "method": (
            "annual_minus_nine_months_same_filing"
            if annual_row["accn"] == nine_row["accn"]
            else "annual_minus_nine_months_cross_filing"
        ),
        "annual_accn": annual_row["accn"],
        "nine_months_accn": nine_row["accn"],
        "nine_months_filed": nine_row["filed"],
        "annual_revenue": annual_row["revenue"],
        "nine_months_revenue": nine_row["revenue"],
    })

derived_q4 = pd.DataFrame(derived_q4_rows)

direct = direct_quarters[
    ["accn", "filed", "start", "end", *metrics, "method"]
].copy()

parts = [part for part in [direct, derived_q4] if not part.empty]
quarter_candidates = pd.concat(parts, ignore_index=True)

quarter_candidates["method_priority"] = (
    quarter_candidates["method"].eq("direct_quarter")
    .map({True: 0, False: 1})
)

quarterly = (
    quarter_candidates
    .sort_values(["end", "filed", "method_priority", "accn"])
    .drop_duplicates(["start", "end"], keep="first")
    .sort_values("end")
    .reset_index(drop=True)
)

quarterly["quarter"] = quarterly["end"].dt.to_period("Q").astype(str)

assert not quarterly["quarter"].duplicated().any()
assert (quarterly["revenue"] > 0).all()
assert (quarterly["filed"] >= quarterly["end"]).all()

quarterly["operating_margin_pct"] = (
    quarterly["operating_income"] / quarterly["revenue"] * 100
)
quarterly["net_margin_pct"] = (
    quarterly["net_income"] / quarterly["revenue"] * 100
)

quarterly["previous_year_end"] = (
    quarterly["end"] - pd.DateOffset(years=1)
)

quarterly = quarterly.merge(
    quarterly[["end", "revenue"]].rename(columns={
        "end": "previous_year_end",
        "revenue": "previous_year_revenue",
    }),
    on="previous_year_end",
    how="left",
    validate="many_to_one",
)

quarterly["revenue_yoy_pct"] = (
    quarterly["revenue"] / quarterly["previous_year_revenue"] - 1
) * 100

quarterly.to_csv(
    PROCESSED / "quarterly_financials.csv", index=False
)
derived_q4.to_csv(
    REPORTS / "derived_q4_provenance.csv", index=False
)

print("Q4 derivados:")
display(
    quarterly.loc[
        quarterly["method"].ne("direct_quarter"),
        [
            "quarter", "filed", "method", "revenue",
            "operating_income", "net_income",
            "annual_accn", "nine_months_accn",
        ],
    ]
)

if q4_issues:
    display(pd.DataFrame(q4_issues))

In [ ]:
if not derived_q4.empty:
    np.testing.assert_allclose(
        derived_q4["revenue"] + derived_q4["nine_months_revenue"],
        derived_q4["annual_revenue"],
        rtol=1e-12,
        atol=0.01,
    )

    assert (
        derived_q4["nine_months_filed"] <= derived_q4["filed"]
    ).all()

expected_quarters = pd.period_range(
    "2018Q1",
    pd.Period(quarterly["end"].max(), freq="Q"),
    freq="Q",
)
observed_quarters = pd.PeriodIndex(quarterly["quarter"], freq="Q")
missing_quarters = expected_quarters.difference(observed_quarters)

coverage = pd.DataFrame({
    "quarter": expected_quarters.astype(str),
    "available": expected_quarters.isin(observed_quarters),
})
coverage.to_csv(REPORTS / "quarterly_financial_coverage.csv", index=False)

print("Trimestres construídos:", len(quarterly))
print("Trimestres ausentes:", list(missing_quarters.astype(str)))
display(quarterly["method"].value_counts().to_frame("quarters"))

display(
    quarterly[
        ["quarter", "filed", "method", "revenue",
         "operating_margin_pct", "net_margin_pct", "revenue_yoy_pct"]
    ].tail(12)
)

In [ ]:
expected_quarters = pd.period_range(
    "2018Q1",
    pd.Period(quarterly["end"].max(), freq="Q"),
    freq="Q",
)

observed_quarters = pd.PeriodIndex(quarterly["quarter"], freq="Q")
missing_quarters = expected_quarters.difference(observed_quarters)

print("Trimestres construídos:", len(quarterly))
print("Trimestres ausentes:", list(missing_quarters.astype(str)))
print("\nMétodos utilizados:")
display(quarterly["method"].value_counts().to_frame("quarters"))

coverage = pd.DataFrame({
    "quarter": expected_quarters.astype(str),
    "available": expected_quarters.isin(observed_quarters),
})
coverage.to_csv(REPORTS / "quarterly_financial_coverage.csv", index=False)

# Reindexação mantém lacunas visíveis nos gráficos.
plot_data = quarterly.set_index("quarter").reindex(
    expected_quarters.astype(str)
)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

axes[0].plot(
    plot_data.index,
    plot_data["revenue"] / 1e9,
    marker="o",
)
axes[0].set_title("Receita trimestral — primeira divulgação completa selecionada")
axes[0].set_ylabel("US$ bilhões")

axes[1].plot(
    plot_data.index,
    plot_data["operating_margin_pct"],
    marker="o",
    label="Margem operacional",
)
axes[1].plot(
    plot_data.index,
    plot_data["net_margin_pct"],
    label="Margem líquida",
)
axes[1].set_ylabel("Margem (%)")
axes[1].legend()

for ax in axes:
    ax.grid(alpha=0.2)

axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.savefig(REPORTS / "quarterly_financials.png", dpi=150)
plt.show()

In [ ]:
quarterly_check = quarterly.copy()
quarterly_check["year"] = quarterly_check["end"].dt.year

quarter_counts = quarterly_check.groupby("year").size()
full_years = quarter_counts.loc[quarter_counts.eq(4)].index

quarter_totals = (
    quarterly_check.loc[quarterly_check["year"].isin(full_years)]
    .groupby("year")[metrics]
    .sum()
    .reset_index()
)

annual_check = annual_selected.copy()
annual_check["year"] = annual_check["end"].dt.year

annual_reconciliation = quarter_totals.merge(
    annual_check[["year", "accn", "filed", *metrics]],
    on="year",
    suffixes=("_quarter_sum", "_annual"),
    validate="one_to_one",
)

for metric in metrics:
    annual_reconciliation[f"{metric}_difference_usd"] = (
        annual_reconciliation[f"{metric}_quarter_sum"]
        - annual_reconciliation[f"{metric}_annual"]
    )

difference_columns = [
    f"{metric}_difference_usd" for metric in metrics
]

annual_reconciliation["within_one_dollar"] = (
    annual_reconciliation[difference_columns].abs().le(1).all(axis=1)
)

display(
    annual_reconciliation[
        ["year", *difference_columns, "within_one_dollar"]
    ]
)

annual_reconciliation.to_csv(
    REPORTS / "annual_quarter_reconciliation.csv", index=False
)

margin_review = quarterly.loc[
    quarterly["net_margin_pct"] > quarterly["operating_margin_pct"],
    ["quarter", "revenue", "operating_income", "net_income",
     "operating_margin_pct", "net_margin_pct", "accn"],
].copy()

print("Trimestres em que o lucro líquido supera o operacional:")
display(margin_review)

margin_review.to_csv(
    REPORTS / "financial_margin_review.csv", index=False
)

In [ ]:
financial_bridges = pd.DataFrame([
    {
        "quarter": "2019Q4",
        "operating_income": 458_512_000,
        "nonoperating_net": -309_179_000,
        "expense_tax": -437_637_000,
        "net_income": 586_970_000,
        "source": (
            "https://www.sec.gov/Archives/edgar/data/"
            "1065280/000106528020000306/ex991q220.htm"
        ),
        "explanation": (
            "Benefício tributário de US$ 437,637 milhões "
            "compensou despesas não operacionais líquidas "
            "de US$ 309,179 milhões."
        ),
    },
    {
        "quarter": "2026Q1",
        "operating_income": 3_956_997_000,
        "nonoperating_net": 2_852_166_000 - 262_077_000,
        "expense_tax": 1_264_295_000,
        "net_income": 5_282_791_000,
        "source": (
            "https://www.sec.gov/Archives/edgar/data/"
            "1065280/000106528026000138/nflx-20260331.htm"
        ),
        "explanation": (
            "Outras receitas incluíram US$ 2,8 bilhões "
            "de taxa de rescisão da transação com WBD. "
            "Esse item não é receita operacional recorrente."
        ),
    },
])

financial_bridges["calculated_net_income"] = (
    financial_bridges["operating_income"]
    + financial_bridges["nonoperating_net"]
    - financial_bridges["expense_tax"]
)

np.testing.assert_allclose(
    financial_bridges["calculated_net_income"],
    financial_bridges["net_income"],
    rtol=0,
    atol=1,
)

# Confere os valores contra a tabela já construída no projeto.
for row in financial_bridges.itertuples():
    quarter_row = quarterly.loc[
        quarterly["quarter"].eq(row.quarter)
    ]
    assert len(quarter_row) == 1

    for metric in ["operating_income", "net_income"]:
        np.testing.assert_allclose(
            quarter_row[metric].iloc[0],
            getattr(row, metric),
            rtol=0,
            atol=1,
        )

financial_bridges.to_csv(
    REPORTS / "financial_margin_explanations.csv",
    index=False,
)



## 5. Calendário de pregões
NASDAQ para NFLX; NYSE como referência para o índice de preços S&P 500. Não preenche nem exclui divergências automaticamente.

In [ ]:
import pandas_market_calendars as mcal
from importlib.metadata import version

calendar_start = pd.Timestamp(START).date()
calendar_end = (
    pd.Timestamp(END_EXCLUSIVE) - pd.Timedelta(days=1)
).date()

calendar_results = []
calendar_discrepancies = []

for ticker, calendar_name in [
    ("NFLX", "NASDAQ"),
    ("^GSPC", "NYSE"),
]:
    calendar = mcal.get_calendar(calendar_name)
    schedule = calendar.schedule(
        start_date=calendar_start,
        end_date=calendar_end,
    )

    # O índice representa datas das sessões, não horários UTC
    # a serem convertidos para o dia anterior.
    expected_dates = pd.DatetimeIndex(schedule.index)
    if expected_dates.tz is not None:
        expected_dates = expected_dates.tz_localize(None)
    expected_dates = expected_dates.normalize()

    observed_dates = pd.DatetimeIndex(
        market_frames[ticker]["Date"].dropna()
    ).normalize()

    missing = expected_dates.difference(observed_dates)
    unexpected = observed_dates.difference(expected_dates)

    calendar_results.append({
        "ticker": ticker,
        "calendar": calendar_name,
        "expected_sessions": len(expected_dates),
        "observed_unique_dates": observed_dates.nunique(),
        "missing_sessions": len(missing),
        "unexpected_dates": len(unexpected),
        "passed": len(missing) == 0 and len(unexpected) == 0,
    })

    for issue, dates in [
        ("missing_session", missing),
        ("unexpected_date", unexpected),
    ]:
        for date in dates:
            calendar_discrepancies.append({
                "ticker": ticker,
                "calendar": calendar_name,
                "issue": issue,
                "date": date,
            })

    schedule.to_csv(
        REPORTS / f"{calendar_name.lower()}_expected_schedule.csv"
    )

calendar_audit = pd.DataFrame(calendar_results)
calendar_details = pd.DataFrame(
    calendar_discrepancies,
    columns=["ticker", "calendar", "issue", "date"],
)

calendar_audit.to_csv(
    REPORTS / "trading_calendar_audit.csv", index=False
)
calendar_details.to_csv(
    REPORTS / "trading_calendar_discrepancies.csv", index=False
)

environment["pandas_market_calendars"] = version(
    "pandas_market_calendars"
)
(REPORTS / "environment.json").write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

display(calendar_audit)

if calendar_details.empty:
    print("Nenhuma divergência contra os calendários selecionados.")
else:
    display(calendar_details)
    print("Não preencher nem remover datas antes de investigar.")



## 6. Exportação e dashboard
CSS e JavaScript incorporados. Cinco painéis com navegação lateral, marca d’água e fontes contábeis.

In [ ]:
def records(frame):
    # Converte datas e transforma NaN em null válido no JSON.
    return json.loads(
        frame.to_json(
            orient="records",
            date_format="iso",
            double_precision=15,
        )
    )

dashboard_payload = {
    "metadata": {
        "project": "NFLX Insight",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "market_start": str(base["Date"].min().date()),
        "market_end": str(base["Date"].max().date()),
        "selected_model": "Baseline_zero",
        "financial_quarters": len(quarterly),
        "direct_quarters": int(
            quarterly["method"].eq("direct_quarter").sum()
        ),
        "derived_quarters": int(
            quarterly["method"].ne("direct_quarter").sum()
        ),
    },
    "market": records(base),
    "performance_summary": records(
        summary.rename_axis("asset").reset_index()
    ),
    "financials": records(quarterly),
    "financial_reconciliation": records(annual_reconciliation),
    "cv_metrics": records(cv_summary),
    "validation_metrics": records(validation_results),
    "test_metrics": records(test_metrics),
    "test_predictions": records(test_output),
    "historical_ab_summary": records(ab_summary),
    "historical_ab_daily": records(execution),
    "historical_ab_uncertainty": records(ab_uncertainty),
    "cost_sensitivity": records(cost_sensitivity),
    "cost_decomposition": records(cost_decomposition),
    "ridge_coefficients": records(
        coefficient_table.drop(columns="absolute_effect")
    ),
    "protocol": frozen_protocol,
    "limitations": [
        "Histórico de preços ajustado retrospectivamente pela fonte.",
        "S&P 500 usado como índice de preços, sem dividendos reinvestidos.",
        "Histórico completo visualizado antes da avaliação final.",
        "Comparação histórica A/B não é experimento causal randomizado.",
        "Custos são cenários assumidos, não despesas reais observadas.",
        "Modelo prevê fechamento a fechamento; estratégia opera abertura a fechamento.",
        "Caixa assume rendimento zero.",
        "Oito Q4 derivados entre documentos, sujeitos a reclassificações.",
        "Data de entrega à SEC não equivale à data do anúncio dos resultados.",
    ],
}

dashboard_payload["financial_margin_explanations"] = records(financial_bridges)
dashboard_payload["trading_calendar_audit"] = records(calendar_audit)
dashboard_payload["trading_calendar_discrepancies"] = records(calendar_details)

EXPORTS = ROOT / "exports"
EXPORTS.mkdir(parents=True, exist_ok=True)

payload_text = json.dumps(
    dashboard_payload,
    ensure_ascii=False,
    allow_nan=False,
    separators=(",", ":"),
)

payload_path = EXPORTS / "dashboard_data.json"
payload_path.write_text(payload_text, encoding="utf-8")

# Verifica o arquivo efetivamente gravado.
loaded_payload = json.loads(payload_path.read_text(encoding="utf-8"))

assert len(loaded_payload["market"]) == len(base)
assert len(loaded_payload["financials"]) == 34
assert len(loaded_payload["test_predictions"]) == len(test)
assert loaded_payload["metadata"]["selected_model"] == "Baseline_zero"

print("Dados do dashboard exportados e verificados.")
print("Tamanho:", round(payload_path.stat().st_size / 1024, 1), "KB")

In [ ]:
from google.colab import drive
import shutil
drive.mount("/content/drive")
delivery_folder = Path("/content/drive/MyDrive/Netflix-stock-analysis/dashboard")
delivery_folder.mkdir(parents=True, exist_ok=True)


In [ ]:
import html
import plotly
import plotly.graph_objects as go

D = loaded_payload
figures = []

def chart(traces, title, y_title="", percent=False):
    fig = go.Figure(traces)
    fig.update_layout(
        template="plotly_dark",
        title=dict(text=title, font=dict(size=16)),
        paper_bgcolor="#151519",
        plot_bgcolor="#151519",
        font=dict(family="Arial", color="#eeeeef"),
        height=330,
        margin=dict(l=55, r=25, t=60, b=45),
        legend=dict(orientation="h", y=1.15),
        yaxis_title=y_title,
        hovermode="x unified",
        colorway=["#e50914", "#b9becb", "#ffb454"],
    )
    if percent:
        fig.update_yaxes(tickformat=".1%")
    figures.append(fig)
    return len(figures) - 1

market = pd.DataFrame(D["market"])
dates = market["Date"]

overview_chart = chart([
    go.Scatter(
        x=dates,
        y=market["Adj Close"] / market["Adj Close"].iloc[0] * 100,
        name="NFLX",
    ),
    go.Scatter(
        x=dates,
        y=market["SP500_Adj_Close"]
          / market["SP500_Adj_Close"].iloc[0] * 100,
        name="S&P 500",
    ),
], "Evolução dos preços · base 100", "Índice")

risk_chart = chart([
    go.Scatter(x=dates, y=market["NFLX_Drawdown"], name="NFLX"),
    go.Scatter(x=dates, y=market["SP500_Drawdown"], name="S&P 500"),
], "Queda desde o maior preço observado", "Drawdown", percent=True)

financial = pd.DataFrame(D["financials"])

revenue_chart = chart([
    go.Bar(
        x=financial["quarter"],
        y=financial["revenue"] / 1e9,
        name="Receita",
        customdata=financial["method"],
        hovertemplate=(
            "%{x}<br>US$ %{y:.2f} bilhões"
            "<br>%{customdata}<extra></extra>"
        ),
    )
], "Receita trimestral", "US$ bilhões")

margin_chart = chart([
    go.Scatter(
        x=financial["quarter"],
        y=financial["operating_margin_pct"],
        name="Operacional",
    ),
    go.Scatter(
        x=financial["quarter"],
        y=financial["net_margin_pct"],
        name="Líquida",
    ),
], "Margens trimestrais", "%")

validation_metrics = pd.DataFrame(D["validation_metrics"])
final_metrics = pd.DataFrame(D["test_metrics"])

model_chart = chart([
    go.Bar(
        x=validation_metrics["model"],
        y=validation_metrics["MAE_bps"],
        name="Validação · 2024",
    ),
    go.Bar(
        x=final_metrics["model"],
        y=final_metrics["MAE_bps"],
        name="Teste · 2025–2026",
    ),
], "Erro de previsão · menor é melhor", "MAE · pontos-base")

ab_daily = pd.DataFrame(D["historical_ab_daily"])

ab_chart = chart([
    go.Scatter(
        x=ab_daily["target_date"],
        y=ab_daily["wealth_A"],
        name="A · operar diariamente",
    ),
    go.Scatter(
        x=ab_daily["target_date"],
        y=ab_daily["wealth_B"],
        name="B · filtro Ridge",
    ),
    go.Scatter(
        x=ab_daily["target_date"],
        y=[1.0] * len(ab_daily),
        name="Caixa · rendimento zero",
        line=dict(dash="dot"),
    ),
], "Capital acumulado após custos · inicial 1", "Capital")

sensitivity = pd.DataFrame(D["cost_sensitivity"])
cost_chart = chart([
    go.Scatter(
        x=sensitivity.loc[sensitivity["strategy"].eq(s), "cost_bps_per_side"],
        y=sensitivity.loc[sensitivity["strategy"].eq(s), "total_return_pct"],
        mode="lines+markers",
        name=f"Estratégia {s}",
    )
    for s in ["A", "B"]
], "Sensibilidade aos custos · posições fixas", "Retorno acumulado (%)")
figures[cost_chart].update_xaxes(title="Custo por ponta · pontos-base")

def graph(index):
    return figures[index].to_html(
        full_html=False,
        include_plotlyjs=True if index == 0 else False,
        config={"responsive": True, "displaylogo": False},
    )

def table(records, columns):
    frame = pd.DataFrame(records)[columns].copy()
    for column in frame.select_dtypes(include="number"):
        frame[column] = frame[column].map(
            lambda value: "—" if pd.isna(value) else f"{value:.2f}"
        )
    return frame.to_html(index=False, border=0, escape=True, na_rep="—")

def card(label, value, detail):
    return (
        '<div class="card"><span>' + html.escape(label) + '</span>'
        '<strong>' + html.escape(value) + '</strong>'
        '<small>' + html.escape(detail) + '</small></div>'
    )

performance = pd.DataFrame(D["performance_summary"]).set_index("asset")
nflx_stats = performance.loc["NFLX"]
last_financial = financial.iloc[-1]
ab_stats = pd.DataFrame(D["historical_ab_summary"]).set_index("strategy")
uncertainty = D["historical_ab_uncertainty"][0]

overview_cards = (
    card("Retorno NFLX", f'{nflx_stats["Retorno acumulado (%)"]:.2f}%',
         "Histórico completo · preços ajustados")
    + card("Volatilidade", f'{nflx_stats["Volatilidade anualizada (%)"]:.2f}%',
           "Anualizada · 252 sessões")
    + card("Trimestres", str(len(financial)),
           "26 diretos · 8 derivados")
)

limits = "<ul>" + "".join(
    "<li>" + html.escape(item) + "</li>" for item in D["limitations"]
) + "</ul>"

panels = [
    (
        "Visão geral",
        overview_cards
        + graph(overview_chart)
        + '<p class="note">Preço e risco contam histórias diferentes. '
          'O benchmark é o índice de preços do S&P 500, sem dividendos.</p>'
    ),
    (
        "Risco",
        '<div class="cards">'
        + card("Drawdown NFLX",
               f'{nflx_stats["Drawdown máximo (%)"]:.2f}%',
               "Queda máxima desde um pico")
        + card("Pior sessão",
               f'{nflx_stats["Pior sessão (%)"]:.2f}%',
               "Movimento preservado na análise")
        + '</div>'
        + graph(risk_chart)
        + '<p class="note">Movimentos extremos não foram removidos. '
          'Datas alinhadas entre fontes não comprovam, isoladamente, '
          'a completude do calendário da bolsa.</p>'
    ),
    (
        "Fundamentos",
        '<div class="cards">'
        + card("Receita · " + last_financial["quarter"],
               f'US$ {last_financial["revenue"] / 1e9:.2f} bi',
               "Fonte: SEC EDGAR")
        + card("Margem operacional",
               f'{last_financial["operating_margin_pct"]:.2f}%',
               last_financial["quarter"])
        + '</div>'
        + '<div class="charts">' + graph(revenue_chart)
        + graph(margin_chart) + '</div>'
        + '<p class="note">Q4 = ano − nove meses, entre documentos '
          'identificados. Reconciliação anual sem diferenças nos três '
          'indicadores de 2018 a 2025. A margem líquida de 2026Q1 '
          'requer investigação; nenhuma causa foi atribuída.</p>'
    ),
    (
        "Modelos",
        '<div class="cards">'
        + card("Modelo selecionado", "Baseline",
               "Seleção na validação de 2024")
        + card("Alternativa experimental", "Ridge",
               "Não superou o baseline no teste")
        + '</div>'
        + graph(model_chart)
        + '<div class="tablewrap">'
        + table(D["test_metrics"], [
            "model", "MAE_bps", "RMSE_bps",
            "directional_accuracy_pct",
        ])
        + '</div><p class="note">Alvo: retorno do próximo fechamento. '
          'Validação temporal com intervalo de um pregão. '
          'O teste não foi usado para ajustar parâmetros.</p>'
    ),
    (
        "Comparação A/B",
        '<div class="cards">'
        + card("A · retorno líquido",
               f'{ab_stats.loc["A", "net_total_return_pct"]:.2f}%',
               "Operar em todas as sessões")
        + card("B · retorno líquido",
               f'{ab_stats.loc["B", "net_total_return_pct"]:.2f}%',
               "Operar se a previsão superar 10 bps")
        + '</div>'
        + '<div class="charts">' + graph(ab_chart)
        + graph(cost_chart) + '</div>'
        + '<p class="note">Comparação histórica, sem randomização. '
          'Execução: abertura → fechamento. Custo principal: 5 bps '
          'por ponta. Ambas perderam dinheiro nesse cenário.</p>'
        + '<p class="note">Diferença média B−A: '
        + f'{uncertainty["mean_daily_difference_bps"]:.3f} bps/sessão. '
        + 'Intervalo aproximado de 95%: '
        + f'[{uncertainty["approximate_95pct_lower_bps"]:.3f}; '
        + f'{uncertainty["approximate_95pct_upper_bps"]:.3f}]. '
        + 'Inclui zero: vantagem inconclusiva.</p>'
    ),
]

buttons = "".join(
    f'<button class="tab" data-index="{i}" '
    f'aria-pressed="{"true" if i == 0 else "false"}">'
    f'{html.escape(title)}</button>'
    for i, (title, _) in enumerate(panels)
)

sections = "".join(
    f'<section class="panel" aria-label="{html.escape(title)}"'
    + ('' if i == 0 else ' inert')
    + f'><h1>{html.escape(title)}</h1>{content}</section>'
    for i, (title, content) in enumerate(panels)
)

dashboard_html = """<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>NFLX Insight · Dados, risco e evidências</title>
<style>
*{box-sizing:border-box}
body{margin:0;background:#0c0c10;color:#eeeef0;font-family:Arial,sans-serif}
header{padding:18px 24px;border-bottom:1px solid #29292f;
display:flex;justify-content:space-between;gap:12px;align-items:center}
.brand{font-size:19px;font-weight:800;letter-spacing:1px}
.brand b{color:#e50914}
.subtitle{font-size:12px;color:#a9abb5;margin-top:5px}
nav{display:flex;gap:8px;padding:12px 24px;overflow-x:auto}
button{cursor:pointer;font:inherit}
.tab{border:1px solid #34343c;background:#17171c;color:#c8c8d0;
border-radius:8px;padding:10px 15px;white-space:nowrap}
.tab[aria-pressed=true]{background:#e50914;border-color:#e50914;color:white}
button:focus-visible,a:focus-visible{outline:3px solid #ffb454}
.viewport{overflow:hidden}
.track{display:flex;transition:transform .55s cubic-bezier(.22,.68,.2,1)}
.panel{flex:0 0 100%;min-width:0;height:calc(100dvh - 151px);
overflow-y:auto;padding:8px 24px 32px}
h1{font-size:25px;margin:10px 0 18px}
.cards{display:flex;gap:12px;flex-wrap:wrap}
.card{display:inline-flex;vertical-align:top;flex-direction:column;
background:#151519;border:1px solid #2b2b32;border-radius:12px;
padding:17px;min-width:210px;margin:0 10px 14px 0}
.card span,.card small{color:#b0b0bc;font-size:12px}
.card strong{font-size:29px;margin:9px 0}
.plotly-graph-div{border-radius:12px;overflow:hidden;margin:12px 0}
.charts{display:grid;grid-template-columns:1fr 1fr;gap:14px}
.charts>div{min-width:0}
.note{color:#c2c2cc;font-size:13px;line-height:1.6;
border-left:3px solid #e50914;padding-left:12px}
.tablewrap{overflow:auto}
table{width:100%;border-collapse:collapse;font-size:13px}
th,td{text-align:left;padding:12px;border-bottom:1px solid #303039}
details{font-size:13px;color:#bcbcc7}
details ul{line-height:1.7}
.controls{display:flex;gap:8px}
.controls button{background:#191920;border:1px solid #34343c;
color:white;padding:8px 12px;border-radius:7px}
@media(max-width:800px){
header{padding:14px}.panel{padding:8px 14px 24px}
nav{padding:10px 14px}.charts{grid-template-columns:1fr}
.card{min-width:145px;flex:1}.card strong{font-size:24px}
}
@media(prefers-reduced-motion:reduce){.track{transition:none}}
</style>
</head>
<body>
<header>
<div><div class="brand"><b>NFLX</b> INSIGHT</div>
<div class="subtitle">__PERIOD__ · snapshots históricos</div></div>
<div class="controls">
<button id="prev" aria-label="Painel anterior">←</button>
<button id="next" aria-label="Próximo painel">→</button>
</div>
</header>
<nav aria-label="Painéis do dashboard">__BUTTONS__</nav>
<div class="viewport"><main class="track">__SECTIONS__</main></div>
<script>
const track=document.querySelector('.track');
const panels=[...document.querySelectorAll('.panel')];
const tabs=[...document.querySelectorAll('.tab')];
let current=0;
function resizeCharts(){
 document.querySelectorAll('.js-plotly-plot').forEach(
  el=>Plotly.Plots.resize(el)
 );
}
function go(index){
 current=Math.max(0,Math.min(panels.length-1,index));
 track.style.transform=`translateX(-${current*100}%)`;
 tabs.forEach((tab,i)=>tab.setAttribute('aria-pressed',String(i===current)));
 panels.forEach((panel,i)=>panel.inert=i!==current);
 document.getElementById('prev').disabled=current===0;
 document.getElementById('next').disabled=current===panels.length-1;
 resizeCharts();
}
tabs.forEach(tab=>tab.addEventListener('click',()=>go(Number(tab.dataset.index))));
document.getElementById('prev').onclick=()=>go(current-1);
document.getElementById('next').onclick=()=>go(current+1);
document.addEventListener('keydown',event=>{
 if(['INPUT','TEXTAREA','SELECT'].includes(event.target.tagName))return;
 if(event.key==='ArrowRight'){event.preventDefault();go(current+1)}
 if(event.key==='ArrowLeft'){event.preventDefault();go(current-1)}
});
window.addEventListener('resize',resizeCharts);
go(0);
</script>
</body></html>
"""

# Acrescenta as limitações ao painel final.
sections = sections.rsplit("</section>", 1)
sections = (
    sections[0]
    + '<details><summary>Metodologia e limitações</summary>'
    + limits + '</details></section>' + sections[1]
)

dashboard_html = (
    dashboard_html
    .replace("__PERIOD__", html.escape(
        D["metadata"]["market_start"] + " a " + D["metadata"]["market_end"]
    ))
    .replace("__BUTTONS__", buttons)
    .replace("__SECTIONS__", sections)
)

dashboard_path = EXPORTS / "index.html"
dashboard_path.write_text(dashboard_html, encoding="utf-8")

environment["plotly"] = plotly.__version__
(REPORTS / "environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)

# Verificações estruturais; a aparência será conferida na prévia.
assert dashboard_html.count('<section class="panel"') == 5
assert "__SECTIONS__" not in dashboard_html
from html.parser import HTMLParser

class ScriptSourceInspector(HTMLParser):
    def __init__(self):
        super().__init__()
        self.sources = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "script":
            attributes = dict(attrs)
            if attributes.get("src"):
                self.sources.append(attributes["src"])

inspector = ScriptSourceInspector()
inspector.feed(dashboard_html)

assert not inspector.sources, (
    f"Scripts externos encontrados: {inspector.sources}"
)

print("Verificação passou: todos os scripts estão incorporados.")
print("Dashboard gerado:", dashboard_path)
print("Tamanho:", round(dashboard_path.stat().st_size / 1024 / 1024, 2), "MB")

In [ ]:
from IPython.display import HTML, display

enhanced_css = """
<style id="motion-enhancement">
:root{
  --motion-duration:760ms;
  --motion-ease:cubic-bezier(.22,1,.36,1);
}

html{scroll-behavior:smooth}
.viewport{position:relative}

.track{
  align-items:stretch;
  transition:transform var(--motion-duration) var(--motion-ease);
  will-change:transform;
}

.panel{
  scrollbar-width:thin;
  scrollbar-color:#454550 #151519;
}

.panel h1,
.panel .card,
.panel .plotly-graph-div,
.panel .tablewrap,
.panel .note,
.panel details{
  opacity:0;
  transform:translateY(14px);
  transition:
    opacity 480ms ease,
    transform 620ms var(--motion-ease);
}

.panel.is-active h1,
.panel.is-active .card,
.panel.is-active .plotly-graph-div,
.panel.is-active .tablewrap,
.panel.is-active .note,
.panel.is-active details{
  opacity:1;
  transform:translateY(0);
}

.panel.is-active h1{transition-delay:50ms}
.panel.is-active .card{transition-delay:100ms}
.panel.is-active .plotly-graph-div{transition-delay:160ms}
.panel.is-active .note{transition-delay:220ms}

.tab{
  transition:
    background-color 220ms ease,
    border-color 220ms ease,
    color 220ms ease,
    transform 220ms ease,
    box-shadow 220ms ease;
}

.tab:hover{
  background:#25252c;
  border-color:#696974;
  transform:translateY(-2px);
}

.tab[aria-pressed=true]{
  background:#e50914;
  border-color:#e50914;
  box-shadow:0 5px 20px rgba(229,9,20,.18);
}

.card{
  transition:
    opacity 480ms ease,
    transform 620ms var(--motion-ease),
    border-color 220ms ease,
    box-shadow 220ms ease;
}

.card:hover{
  border-color:#555560;
  box-shadow:0 8px 24px rgba(0,0,0,.18);
}

.controls button{
  transition:background 200ms ease,transform 200ms ease,opacity 200ms ease;
}
.controls button:hover:not(:disabled){
  background:#303039;
  transform:translateY(-1px);
}
.controls button:disabled{opacity:.3;cursor:default}

.panel::-webkit-scrollbar{width:7px}
.panel::-webkit-scrollbar-thumb{
  background:#454550;
  border-radius:10px;
}

@media(prefers-reduced-motion:reduce){
  .track{transition:none;will-change:auto}
  .panel h1,.panel .card,.panel .plotly-graph-div,
  .panel .tablewrap,.panel .note,.panel details{
    transition:none;
    transform:none;
  }
  .tab,.controls button{transition:none}
}
</style>
"""

enhanced_js = """
<script id="motion-controller">
(()=>{
  const track=document.querySelector('.track');
  const panels=[...document.querySelectorAll('.panel')];
  const tabs=[...document.querySelectorAll('.tab')];
  const previous=document.getElementById('prev');
  const next=document.getElementById('next');
  const reducedMotion=window.matchMedia('(prefers-reduced-motion:reduce)');

  let current=0;
  let activationFrame=0;

  function resizeActiveCharts(){
    if(!window.Plotly)return;
    panels[current].querySelectorAll('.js-plotly-plot').forEach(
      graph=>Plotly.Plots.resize(graph)
    );
  }

  function navigate(index){
    const target=Math.max(0,Math.min(panels.length-1,index));
    if(target===current)return;

    const focusWasInOldPanel=panels[current].contains(document.activeElement);

    panels[current].classList.remove('is-active');
    current=target;

    tabs.forEach((tab,i)=>{
      tab.setAttribute('aria-pressed',String(i===current));
    });

    panels.forEach((panel,i)=>{
      panel.inert=i!==current;
    });

    previous.disabled=current===0;
    next.disabled=current===panels.length-1;

    track.style.transform=`translateX(-${current*100}%)`;

    cancelAnimationFrame(activationFrame);
    activationFrame=requestAnimationFrame(()=>{
      panels[current].classList.add('is-active');
      resizeActiveCharts();
    });

    if(focusWasInOldPanel){
      tabs[current].focus({preventScroll:true});
    }

    tabs[current].scrollIntoView({
      behavior:reducedMotion.matches?'instant':'smooth',
      block:'nearest',
      inline:'nearest'
    });
  }

  tabs.forEach(tab=>{
    tab.onclick=()=>navigate(Number(tab.dataset.index));
  });

  previous.onclick=()=>navigate(current-1);
  next.onclick=()=>navigate(current+1);

  document.addEventListener('keydown',event=>{
    if(event.defaultPrevented)return;
    if(event.target.closest(
      'input,textarea,select,[contenteditable=true],.js-plotly-plot'
    ))return;

    if(event.key==='ArrowRight'){
      event.preventDefault();
      navigate(current+1);
    }
    if(event.key==='ArrowLeft'){
      event.preventDefault();
      navigate(current-1);
    }
  });

  track.addEventListener('transitionend',event=>{
    if(event.target===track && event.propertyName==='transform'){
      resizeActiveCharts();
    }
  });

  window.addEventListener('resize',resizeActiveCharts);

  panels.forEach((panel,i)=>{
    panel.inert=i!==0;
    panel.classList.toggle('is-active',i===0);
  });

  previous.disabled=true;
  next.disabled=panels.length===1;
  requestAnimationFrame(resizeActiveCharts);
})();
</script>
"""

# Remove somente o controlador antigo da navegação.
# Os scripts dos gráficos Plotly permanecem intactos.
old_controller_start = dashboard_html.find(
    "<script>\nconst track=document.querySelector('.track');"
)

if old_controller_start >= 0:
    old_controller_end = dashboard_html.find(
        "</script>", old_controller_start
    )
    if old_controller_end < 0:
        raise ValueError("Não foi possível localizar o fim do controlador.")

    dashboard_html = (
        dashboard_html[:old_controller_start]
        + dashboard_html[old_controller_end + len("</script>"):]
    )
elif 'id="motion-controller"' not in dashboard_html:
    raise ValueError("Controlador antigo não encontrado; HTML não alterado.")

# Permite repetir esta célula sem duplicar CSS ou eventos.
for marker, closing_tag in [
    ('<style id="motion-enhancement">', "</style>"),
    ('<script id="motion-controller">', "</script>"),
]:
    start = dashboard_html.find(marker)
    if start >= 0:
        end = dashboard_html.find(closing_tag, start)
        dashboard_html = (
            dashboard_html[:start]
            + dashboard_html[end + len(closing_tag):]
        )

dashboard_html = dashboard_html.replace(
    "</head>", enhanced_css + "\n</head>"
).replace(
    "</body>", enhanced_js + "\n</body>"
)

assert dashboard_html.count('id="motion-controller"') == 1
assert dashboard_html.count('id="motion-enhancement"') == 1
assert dashboard_html.count('<section class="panel"') == 5

dashboard_path.write_text(dashboard_html, encoding="utf-8")
shutil.copy2(dashboard_path, delivery_folder / "index.html")

display(HTML(
    '<iframe title="NFLX Insight atualizado" '
    'style="width:100%;height:850px;border:0;border-radius:12px" '
    'srcdoc="' + html.escape(dashboard_html, quote=True) + '"></iframe>'
))

print("HTML atualizado e salvo no Drive.")

In [ ]:
import re
import html
import shutil
from IPython.display import HTML, display

netflix_css = """
<style id="netflix-theme">
body{
  background:#141414;
}

/* Marca, menus e controles na mesma barra */
header{
  min-height:86px;
  padding:0 4%;
  border-bottom:none;
  background:linear-gradient(180deg,#080808 0%,#141414 100%);
  display:flex;
  align-items:center;
  gap:34px;
  position:relative;
  z-index:5;
}

header>div:first-child{
  flex-shrink:0;
}

.brand{
  font-size:30px;
  font-weight:900;
  letter-spacing:-1.4px;
  line-height:1;
  white-space:nowrap;
}

.brand b{
  color:#e50914;
}

.brand .brand-detail{
  color:#eee;
  font-size:12px;
  letter-spacing:3px;
  display:block;
  margin-top:8px;
}

.subtitle{
  font-size:10px;
  color:#929292;
  margin-top:8px;
}

/* Navegação discreta, próxima da Netflix */
header nav{
  display:flex;
  align-items:center;
  flex:1;
  gap:24px;
  padding:0;
  overflow-x:auto;
  scrollbar-width:none;
}

header nav::-webkit-scrollbar{
  display:none;
}

.tab,
.tab[aria-pressed=true]{
  padding:12px 0;
  border:0;
  border-radius:0;
  background:transparent;
  box-shadow:none;
  color:#a7a7a7;
  font-size:13px;
  position:relative;
  transform:none;
}

.tab:hover{
  background:transparent;
  border:0;
  color:#fff;
  transform:none;
}

.tab[aria-pressed=true]{
  color:#fff;
  font-weight:700;
}

.tab::after{
  content:"";
  position:absolute;
  left:0;
  right:0;
  bottom:3px;
  height:2px;
  background:#e50914;
  transform:scaleX(0);
  transform-origin:left;
  transition:transform 350ms cubic-bezier(.22,1,.36,1);
}

.tab[aria-pressed=true]::after{
  transform:scaleX(1);
}

.controls{
  margin-left:auto;
  flex-shrink:0;
}

.controls button{
  width:36px;
  height:36px;
  padding:0;
  border-radius:50%;
  background:transparent;
  border:1px solid #494949;
}

/* Marca d'água atrás do conteúdo */
.netflix-watermark{
  position:fixed;
  right:3%;
  top:50%;
  transform:translateY(-42%);
  width:clamp(180px,25vw,390px);
  height:auto;
  opacity:.035;
  pointer-events:none;
  user-select:none;
  z-index:0;
}

.viewport{
  position:relative;
  z-index:1;
}

.panel{
  height:calc(100dvh - 86px);
  padding:20px 4% 36px;
  background:transparent;
}

.panel h1{
  font-size:30px;
  font-weight:700;
  letter-spacing:-.7px;
  margin-bottom:24px;
}

.card{
  background:rgba(24,24,24,.92);
  border:1px solid #303030;
  border-radius:8px;
}

.card strong{
  font-weight:700;
}

.note{
  color:#b5b5b5;
}

/* Duas linhas no celular, mantendo os menus acessíveis */
@media(max-width:900px){
  header{
    min-height:120px;
    padding:18px 4% 10px;
    flex-wrap:wrap;
    gap:14px;
  }

  .brand{
    font-size:25px;
  }

  header nav{
    order:3;
    flex-basis:100%;
    gap:22px;
  }

  .controls{
    margin-left:auto;
  }

  .panel{
    height:calc(100dvh - 120px);
    padding-top:16px;
  }

  .panel h1{
    font-size:25px;
  }

  .netflix-watermark{
    width:210px;
    right:-25px;
    opacity:.025;
  }
}

@media(prefers-reduced-motion:reduce){
  .tab::after{
    transition:none;
  }
}
</style>
"""

# N vetorial: incorporado ao HTML, sem imagem externa.
netflix_watermark = """
<svg id="netflix-watermark" class="netflix-watermark"
     viewBox="0 0 180 320"
     aria-hidden="true" focusable="false">
  <path fill="#b20710"
        d="M0 0H55V312L0 320Z"/>
  <path fill="#b20710"
        d="M125 0H180V320L125 312Z"/>
  <path fill="#e50914"
        d="M0 0H55L180 320L125 312Z"/>
</svg>
"""

updated_html = dashboard_html

# Remove versões anteriores desta personalização.
updated_html = re.sub(
    r'<style id="netflix-theme">.*?</style>',
    "",
    updated_html,
    flags=re.DOTALL,
)
updated_html = re.sub(
    r'<svg id="netflix-watermark".*?</svg>',
    "",
    updated_html,
    flags=re.DOTALL,
)

# Atualiza a marca.
updated_html, brand_count = re.subn(
    r'<div class="brand">.*?</div>',
    '<div class="brand"><b>NETFLIX</b>'
    '<span class="brand-detail">STOCK INSIGHT</span></div>',
    updated_html,
    count=1,
    flags=re.DOTALL,
)

if brand_count != 1:
    raise ValueError("Marca do dashboard não encontrada.")

# Move o menu para dentro do cabeçalho.
nav_match = re.search(
    r'<nav\b[^>]*>.*?</nav>',
    updated_html,
    flags=re.DOTALL,
)

if not nav_match:
    raise ValueError("Menu do dashboard não encontrado.")

nav_html = nav_match.group(0)
updated_html = (
    updated_html[:nav_match.start()]
    + updated_html[nav_match.end():]
)

header_match = re.search(
    r'<header\b[^>]*>.*?</header>',
    updated_html,
    flags=re.DOTALL,
)

if not header_match:
    raise ValueError("Cabeçalho do dashboard não encontrado.")

header_html = header_match.group(0)
controls_position = header_html.find('<div class="controls">')

if controls_position < 0:
    raise ValueError("Controles de navegação não encontrados.")

header_html = (
    header_html[:controls_position]
    + nav_html
    + header_html[controls_position:]
)

updated_html = (
    updated_html[:header_match.start()]
    + header_html
    + updated_html[header_match.end():]
)

updated_html = updated_html.replace(
    "</head>", netflix_css + "\n</head>", 1
)
updated_html = updated_html.replace(
    "<body>", "<body>\n" + netflix_watermark, 1
)

# Mantém os cinco painéis e o controlador de animação.
assert updated_html.count('<section class="panel"') == 5
assert updated_html.count('id="motion-controller"') == 1
assert updated_html.count('id="netflix-theme"') == 1
assert updated_html.count('id="netflix-watermark"') == 1

dashboard_html = updated_html
dashboard_path.write_text(dashboard_html, encoding="utf-8")

delivery_folder.mkdir(parents=True, exist_ok=True)
shutil.copy2(dashboard_path, delivery_folder / "index.html")

display(HTML(
    '<iframe title="Netflix Stock Insight" '
    'style="width:100%;height:850px;border:0;border-radius:12px" '
    'srcdoc="' + html.escape(dashboard_html, quote=True) + '"></iframe>'
))

print("Visual atualizado e HTML salvo no Drive.")

In [ ]:
explanation_html = """
<div id="financial-explanations">
<p class="note">
<strong>2019Q4 · efeito tributário:</strong>
um benefício de imposto de US$ 437,637 milhões compensou
despesas não operacionais líquidas de US$ 309,179 milhões.
O lucro líquido superou o operacional.
<a href="https://www.sec.gov/Archives/edgar/data/1065280/000106528020000306/ex991q220.htm"
target="_blank" rel="noopener noreferrer"
style="color:#ffb454">Fonte SEC</a>
</p>
<p class="note">
<strong>2026Q1 · item não recorrente:</strong>
outras receitas incluíram US$ 2,8 bilhões de taxa de rescisão
da transação com WBD. A margem líquida elevada não deve ser
interpretada como margem operacional recorrente.
<a href="https://www.sec.gov/Archives/edgar/data/1065280/000106528026000138/nflx-20260331.htm"
target="_blank" rel="noopener noreferrer"
style="color:#ffb454">Fonte SEC</a>
</p>
</div>
"""

# Permite executar novamente sem duplicar as explicações.
dashboard_html = re.sub(
    r'<div id="financial-explanations">.*?</div>',
    "",
    dashboard_html,
    flags=re.DOTALL,
)

heading = "<h1>Fundamentos</h1>"
assert dashboard_html.count(heading) == 1

dashboard_html = dashboard_html.replace(
    heading,
    heading + explanation_html,
    1,
)

dashboard_html = dashboard_html.replace(
    "A margem líquida de 2026Q1 "
    "requer investigação; nenhuma causa foi atribuída.",
    "As diferenças entre margens foram investigadas "
    "e estão explicadas com fontes SEC neste painel.",
)

dashboard_path.write_text(dashboard_html, encoding="utf-8")
shutil.copy2(dashboard_path, delivery_folder / "index.html")
shutil.copy2(payload_path, delivery_folder / "dashboard_data.json")

display(financial_bridges[
    ["quarter", "operating_income", "nonoperating_net",
     "expense_tax", "net_income"]
])

print("Reconciliações passaram; explicações adicionadas ao dashboard.")

In [ ]:
# Consolida exportações e versões após todas as etapas.
from importlib.metadata import version
packages = ["numpy", "pandas", "requests", "matplotlib", "scikit-learn", "plotly", "pandas_market_calendars"]
requirements = "\n".join(f"{package}=={version(package)}" for package in packages) + "\n"
(ROOT / "requirements-run.txt").write_text(requirements, encoding="utf-8")
shutil.copy2(payload_path, delivery_folder / "dashboard_data.json")
shutil.copy2(dashboard_path, delivery_folder / "index.html")
checkpoint_time = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
checkpoint_path = Path("/content/drive/MyDrive/Netflix-stock-analysis/checkpoints") / checkpoint_time
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(ROOT, checkpoint_path)
print("Checkpoint final salvo:", checkpoint_path)
print("Salve também este notebook pelo menu Arquivo do Colab.")
